# Lab Day 10: Fine-tune Qwen3-VL-4B

**Hoàng Văn Nam — 2A202602853**

Notebook hoàn chỉnh E0–E4. E5 tùy chọn không thực hiện. Số liệu báo cáo được sinh từ lần chạy thật; chưa có evidence thì không coi bài đã hoàn tất. Không sửa scripts/tests của đề.


## 0. Môi trường (Colab T4)

In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess
drive.mount('/content/drive')
ROOT=Path('/content/day10-vlm')
SAVE=Path('/content/drive/MyDrive/Day10_VLM')
SAVE.mkdir(parents=True,exist_ok=True)
if not ROOT.exists():
    subprocess.run(['git','clone','https://github.com/namhv521/K4-Track-4-Day10-VisionLanguageModel.git',str(ROOT)],check=True)
os.chdir(ROOT)
%pip install -r requirements-colab.txt pytest
%pip uninstall -y torchao


ValueError: mount failed

In [ ]:
import torch, transformers, peft
assert torch.cuda.is_available(), 'Chọn Runtime > Change runtime type > T4 GPU'
print('Hoàng Văn Nam — 2A202602853')
print(torch.cuda.get_device_name(0),torch.__version__,transformers.__version__,peft.__version__)
subprocess.run(['python','-m','pip','freeze'],stdout=(SAVE/'environment.txt').open('w'),check=True)


## 1. Dữ liệu: tải và chia split

In [ ]:
import urllib.request, zipfile
dataset_zip=Path('/content/road_damage.zip')
if not dataset_zip.exists():
    urllib.request.urlretrieve('https://www.kaggle.com/api/v1/datasets/download/lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes',dataset_zip)
assert zipfile.is_zipfile(dataset_zip), 'Kaggle không trả ZIP: dùng ZIP tải tay và --zip'
result=subprocess.run(['python','scripts/prepare_data.py','--zip',str(dataset_zip)],capture_output=True,text=True)
print(result.stdout,result.stderr)
assert result.returncode==0
result=subprocess.run(['python','-m','pytest','-q'],capture_output=True,text=True)
print(result.stdout,result.stderr)
assert result.returncode==0


In [ ]:
import json
from PIL import Image, ImageDraw
from IPython.display import display
meta=json.loads(Path('data/splits/meta.json').read_text())
print(json.dumps(meta['counts'],indent=2))
rows={s:[json.loads(x) for x in Path(f'data/splits/{s}.jsonl').read_text().splitlines()] for s in ('train','val','test')}
sessions={s:{r['session'] for r in v} for s,v in rows.items()}
assert not sessions['train'] & sessions['val']
assert not sessions['train'] & sessions['test']
assert not sessions['val'] & sessions['test']
for row in rows['train'][:6]:
    im=Image.open(Path('data')/row['image_path']).convert('RGB'); draw=ImageDraw.Draw(im)
    for obj in row['objects']:
        xc,yc,w,h=obj['yolo']
        box=((xc-w/2)*im.width,(yc-h/2)*im.height,(xc+w/2)*im.width,(yc+h/2)*im.height)
        draw.rectangle(box,outline='lime',width=2); draw.text(box[:2],obj['label'],fill='lime')
    display(im)


## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
%%writefile student_lab.py
"""Day 10 student implementation: Hoang Van Nam, 2A202602853.

Instructor scripts and test definitions are left unchanged.
"""
import gc
import hashlib
import json
import math
import random
import re
import shutil
import subprocess
import time
from pathlib import Path

MODEL_ID = 'Qwen/Qwen3-VL-4B-Instruct'
STUDENT_NAME = 'Hoàng Văn Nam'
STUDENT_ID = '2A202602853'
PROMPT = ('Locate every pothole, crack and manhole in this road image. '
          'Return ONLY a JSON list: [{"label":"pothole|crack|manhole",'
          '"bbox_2d":[x1,y1,x2,y2]}]. Coordinates must be integers on '
          'the normalized 0-1000 scale, relative to image width and height. '
          'Use tight boxes. Return [] if none. No explanations or markdown.')
SAVE = Path('/content/drive/MyDrive/Day10_VLM')


def yolo_to_box(label, yolo):
    xc, yc, w, h = yolo
    return {'label': label, 'bbox_2d': [max(0, min(1000, round(v * 1000)))
            for v in (xc-w/2, yc-h/2, xc+w/2, yc+h/2)]}


def objects_to_answer(objects):
    boxes = [yolo_to_box(o['label'], o['yolo']) for o in objects]
    boxes.sort(key=lambda b: (b['bbox_2d'][1], b['bbox_2d'][0]))
    return json.dumps(boxes, separators=(',', ':'))


def dump(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding='utf-8')
    tmp.replace(path)


def read_rows(split):
    from scripts.labcommon import read_jsonl
    return read_jsonl(f'data/splits/{split}.jsonl')


def backup():
    SAVE.mkdir(parents=True, exist_ok=True)
    for name in ('report', 'data/splits'):
        if Path(name).exists():
            shutil.copytree(name, SAVE / name, dirs_exist_ok=True)
    for name in ('student_lab.py', 'student_checks.py', 'REPORT.md', 'lab_day10.ipynb'):
        if Path(name).exists():
            shutil.copy2(name, SAVE / name)


def user_message(row):
    from PIL import Image
    with Image.open(Path('data') / row['image_path']) as im:
        image = im.convert('RGB')
    return {'role': 'user', 'content': [
        {'type': 'image', 'image': image}, {'type': 'text', 'text': PROMPT}]}


def load_base():
    import torch
    from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration
    processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=1024, max_pixels=245760)
    processor.tokenizer.padding_side = 'left'
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
        llm_int8_skip_modules=['model.visual', 'lm_head'])
    model = Qwen3VLForConditionalGeneration.from_pretrained(MODEL_ID,
        quantization_config=quant, dtype=torch.float16,
        device_map={'': 0}, attn_implementation='sdpa')
    assert not any(m.__class__.__name__ == 'Linear4bit' for n, m in model.named_modules()
                   if n.startswith('model.visual.')), 'Visual was quantized'
    groups = {'visual': 0, 'projector': 0, 'language_and_head': 0}
    # Parameter numel on packed 4-bit weights is not the original dense size.
    for n, p in model.named_parameters():
        size = p.numel()
        if p.__class__.__name__ == 'Params4bit' and p.quant_state is not None:
            size = math.prod(p.quant_state.shape)
        key = 'visual' if '.visual.' in n else 'language_and_head'
        groups[key] += size
        if is_projector(n):
            groups['projector'] += size
    dump('work/architecture.json', groups)
    print('Dense architecture parameters:', groups, flush=True)
    return model, processor


def encode_example(processor, row):
    import torch
    user = user_message(row)
    prompt = processor.apply_chat_template([user], tokenize=True,
        add_generation_prompt=True, return_dict=True, return_tensors='pt')
    full = processor.apply_chat_template([user,
        {'role': 'assistant', 'content': [{'type': 'text', 'text': objects_to_answer(row['objects'])}]}],
        tokenize=True, add_generation_prompt=False, return_dict=True, return_tensors='pt')
    n = prompt['input_ids'].shape[1]
    assert torch.equal(full['input_ids'][:, :n], prompt['input_ids']), 'Chat prefix mismatch'
    full['labels'] = full['input_ids'].clone()
    full['labels'][:, :n] = -100
    assert (full['labels'] != -100).any(), 'No supervised tokens'
    return dict(full)


def collate(batch, pad_token_id):
    import torch
    import torch.nn.functional as F
    n = max(x['input_ids'].shape[1] for x in batch)
    out = {}
    for key in batch[0]:
        if key in ('pixel_values', 'image_grid_thw'):
            out[key] = torch.cat([x[key] for x in batch], dim=0)
        elif key in ('input_ids', 'attention_mask', 'labels', 'mm_token_type_ids', 'token_type_ids'):
            pad = -100 if key == 'labels' else pad_token_id if key == 'input_ids' else 0
            out[key] = torch.cat([F.pad(x[key], (0, n-x[key].shape[1]), value=pad) for x in batch])
        else:
            raise ValueError(f'Unsupported processor field: {key}')
    return out


def is_projector(name):
    return 'visual.merger' in name or 'visual.deepstack_merger_list' in name


def apply_setup(model, setup, rank=16):
    import torch
    from peft import LoraConfig, get_peft_model
    llm = r'.*language_model\.layers\.\d+\.(?:self_attn\.(?:q|k|v|o)_proj|mlp\.(?:gate|up|down)_proj)'
    vit = r'.*visual\.blocks\.\d+\.(?:attn\.(?:qkv|proj)|mlp\.(?:linear_fc1|linear_fc2))'
    pattern = f'(?:{llm}|{vit})' if setup == 'E3' else llm
    matched = [n for n, _ in model.named_modules() if re.fullmatch(pattern, n)]
    expected = 348 if setup == 'E3' else 252
    assert len(matched) == expected, (len(matched), expected, matched[:5])
    for p in model.parameters():
        p.requires_grad = False
    model.enable_input_require_grads()
    model = get_peft_model(model, LoraConfig(r=rank, lora_alpha=2*rank,
        lora_dropout=0.05, target_modules=pattern, task_type='CAUSAL_LM'))
    if setup in ('E2', 'E3'):
        for name, p in model.named_parameters():
            if is_projector(name):
                assert p.__class__.__name__ != 'Params4bit', 'Projector is quantized'
                p.data = p.data.to(torch.float32)
                p.requires_grad = True
    # PEFT adapters must be float32 for fp16 GradScaler unscale.
    for name, p in model.named_parameters():
        if p.requires_grad:
            p.data = p.data.float()
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
    model.config.use_cache = False
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(setup, 'LoRA modules:', len(matched), 'trainable_params:', n, flush=True)
    return model, n


def move_batch(batch):
    import torch
    return {k: v.to('cuda', dtype=torch.float16 if k == 'pixel_values' else v.dtype)
            for k, v in batch.items()}


def checkpoint(model, folder, state):
    import torch
    folder.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(folder / 'adapter')
    extra = {n: p.detach().cpu() for n, p in model.named_parameters()
             if p.requires_grad and is_projector(n)}
    if extra:
        torch.save(extra, folder / 'projector.pt')
    tmp = folder / 'trainer.tmp.pt'
    torch.save(state, tmp)
    tmp.replace(folder / 'trainer.pt')
    dump(folder / 'run.json', state['run'])


def amp_optimizer_step(opt, params, scaler):
    """Never update weights with non-finite gradients; lower scale and retry."""
    import torch
    scaler.unscale_(opt)
    norm = torch.nn.utils.clip_grad_norm_(params, 1.0, error_if_nonfinite=False)
    if not torch.isfinite(norm):
        scaler.update(new_scale=scaler.get_scale() * scaler.get_backoff_factor())
        opt.zero_grad(set_to_none=True)
        return False, float(norm)
    scaler.step(opt)
    scaler.update()
    opt.zero_grad(set_to_none=True)
    return True, float(norm)


def train(model, processor, setup, name, rank=16, fraction=1.0, steps=300):
    import torch
    from transformers import get_cosine_schedule_with_warmup
    folder = SAVE / 'runs' / name.lower()
    folder.mkdir(parents=True, exist_ok=True)
    model, nparams = apply_setup(model, setup, rank)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=2e-4, weight_decay=0)
    sched = get_cosine_schedule_with_warmup(opt, math.ceil(steps*0.05), steps)
    scaler = torch.amp.GradScaler('cuda')
    rows = read_rows('train')
    random.Random(42).shuffle(rows)
    rows = rows[:max(1, int(len(rows)*fraction))]
    run = dict(name=name, setup=setup, rank=rank, train_fraction=fraction, seed=42,
               trainable_params=nparams, batch_size=1, grad_accum=8, max_steps=steps,
               optimizer='torch.optim.AdamW', lr=2e-4, completed_steps=0,
               train_minutes=0.0, peak_vram_gb=0.0, train_images=len(rows))
    start_step = 0
    saved = folder / 'trainer.pt'
    if saved.exists():
        state = torch.load(saved, map_location='cpu', weights_only=False)
        old = state['run']
        for key in ('name', 'setup', 'rank', 'train_fraction', 'max_steps'):
            assert old[key] == run[key], f'Resume mismatch: {key}'
        from peft import load_peft_weights, set_peft_model_state_dict
        set_peft_model_state_dict(model, load_peft_weights(str(folder/'adapter'), device='cpu'))
        restore_projector(model, folder)
        opt.load_state_dict(state['optimizer'])
        sched.load_state_dict(state['scheduler'])
        scaler.load_state_dict(state['scaler'])
        run = old
        start_step = old['completed_steps']
        torch.set_rng_state(state['rng'])
        torch.cuda.set_rng_state_all(state['cuda_rng'])
        print('Resume', name, start_step, flush=True)
        del state
    torch.cuda.reset_peak_memory_stats()
    begin = time.perf_counter()
    previous_minutes = run['train_minutes']
    cache = {}
    model.train()
    opt.zero_grad(set_to_none=True)
    log = folder / 'loss.jsonl'
    step, overflow_retries = start_step, 0
    while step < steps:
        loss_sum = 0.0
        for micro in range(8):
            idx = (step*8 + micro) % len(rows)
            if idx not in cache:
                cache[idx] = encode_example(processor, rows[idx])
            batch = move_batch(collate([cache[idx]], processor.tokenizer.pad_token_id))
            with torch.autocast('cuda', dtype=torch.float16):
                loss = model(**batch).loss
            if not torch.isfinite(loss):
                raise FloatingPointError(f'{name} non-finite loss at {step+1}')
            loss_sum += float(loss.detach()) / 8
            scaler.scale(loss / 8).backward()
            del batch, loss
        scale_before = scaler.get_scale()
        updated, norm = amp_optimizer_step(opt, params, scaler)
        if not updated:
            overflow_retries += 1
            run['amp_overflow_skips'] = run.get('amp_overflow_skips', 0) + 1
            event = dict(event='amp_overflow_retry', step=step+1, retry=overflow_retries,
                         scale_before=scale_before, scale_after=scaler.get_scale(),
                         image_ids=[rows[(step*8+i) % len(rows)]['image_id'] for i in range(8)])
            with (folder / 'amp_events.jsonl').open('a', encoding='utf-8') as f:
                f.write(json.dumps(event)+'\n')
            print(name, json.dumps(event), flush=True)
            if overflow_retries >= 8:
                raise FloatingPointError(f'{name}: gradients remain non-finite after 8 retries; see amp_events.jsonl')
            continue
        overflow_retries = 0
        sched.step()
        run.update(completed_steps=step+1,
            train_minutes=previous_minutes+(time.perf_counter()-begin)/60,
            peak_vram_gb=max(run['peak_vram_gb'], torch.cuda.max_memory_allocated()/2**30))
        record = dict(step=step+1, loss=loss_sum, grad_norm=float(norm),
                      lr=sched.get_last_lr()[0], minutes=run['train_minutes'])
        with log.open('a', encoding='utf-8') as f:
            f.write(json.dumps(record)+'\n')
        if (step+1) % 10 == 0 or step == start_step:
            print(name, json.dumps(record), 'VRAM', round(run['peak_vram_gb'], 2), flush=True)
            dump(folder / 'progress.json', run)
        if (step+1) % 50 == 0 or step+1 == steps:
            checkpoint(model, folder, dict(run=run, optimizer=opt.state_dict(),
                scheduler=sched.state_dict(), scaler=scaler.state_dict(),
                rng=torch.get_rng_state(), cuda_rng=torch.cuda.get_rng_state_all()))
        step += 1
    dump(folder / 'run.json', run)
    del cache, opt, sched, scaler, params
    gc.collect()
    torch.cuda.empty_cache()
    return model, run


def restore_projector(model, folder):
    import torch
    path = Path(folder) / 'projector.pt'
    if path.exists():
        values = torch.load(path, map_location='cpu', weights_only=True)
        named = dict(model.named_parameters())
        for name, value in values.items():
            assert name in named, f'Missing projector key {name}'
            named[name].data = value.to(device=named[name].device, dtype=torch.float32)
        print('Restored projector tensors:', len(values), flush=True)


def load_trained(setup, name):
    from peft import PeftModel
    model, processor = load_base()
    folder = SAVE / 'runs' / name.lower()
    model = PeftModel.from_pretrained(model, folder / 'adapter')
    restore_projector(model, folder)
    return model, processor


def infer(model, processor, name, split='test', limit=None, batch_size=2):
    import torch
    from scripts.labcommon import read_jsonl
    rows = read_rows(split)
    if limit:
        rows = rows[:limit]
    folder = SAVE / 'predictions'
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / f'{name.lower()}_{split}.jsonl'
    signature = {'dataset_version': json.loads(Path('data/splits/meta.json').read_text())['dataset_version'],
                 'name': name, 'split': split, 'limit': limit, 'prompt': PROMPT,
                 'model_revision': getattr(model.config, '_commit_hash', None)}
    manifest = path.with_suffix('.manifest.json')
    if manifest.exists():
        assert json.loads(manifest.read_text()) == signature, 'Prediction resume provenance mismatch'
    else:
        assert not path.exists(), 'Existing predictions have no provenance manifest'
        dump(manifest, signature)
    done = {}
    if path.exists():
        for r in read_jsonl(path):
            done[r['image_id']] = r['raw']
    remaining = [r for r in rows if r['image_id'] not in done]
    model.eval()
    model.gradient_checkpointing_disable()
    model.config.use_cache = True
    begin = time.perf_counter()
    with path.open('a', encoding='utf-8') as f:
        for start in range(0, len(remaining), batch_size):
            batch = remaining[start:start+batch_size]
            chats = [[user_message(r)] for r in batch]
            inputs = processor.apply_chat_template(chats, tokenize=True, add_generation_prompt=True,
                return_dict=True, return_tensors='pt', padding=True)
            inputs = move_batch(dict(inputs))
            with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
                output = model.generate(**inputs, max_new_tokens=256, do_sample=False)
            answers = processor.batch_decode(output[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)
            for row, raw in zip(batch, answers):
                f.write(json.dumps({'image_id': row['image_id'], 'raw': raw}, ensure_ascii=False)+'\n')
                done[row['image_id']] = raw
            f.flush()
            if start % 20 == 0 or start+batch_size >= len(remaining):
                print(name, split, f'{len(done)}/{len(rows)}', 'minutes', round((time.perf_counter()-begin)/60,2),
                      'sample', answers[0][:160], flush=True)
            del inputs, output
    assert all(r['image_id'] in done for r in rows)
    return path, {r['image_id']: done[r['image_id']] for r in rows}


def score(path, name, run):
    args = ['python', 'scripts/score.py', '--pred', str(path), '--name', name,
            '--out', f'report/evidence/{name.lower()}.json']
    for key in ('trainable_params', 'peak_vram_gb', 'train_minutes'):
        args += ['--meta', f'{key}={run[key]}']
    result = subprocess.run(args, capture_output=True, text=True, check=True)
    print(result.stdout, flush=True)
    backup()


def release(model):
    import torch
    del model
    gc.collect()
    torch.cuda.empty_cache()


def preflight(model, processor):
    import torch
    rows = read_rows('train')
    sample = encode_example(processor, rows[0])
    image_token = processor.tokenizer.convert_tokens_to_ids('<|image_pad|>')
    count = int((sample['input_ids'] == image_token).sum())
    print('Image tokens:', count)
    assert count == 220, count
    text = processor.tokenizer.decode(sample['labels'][sample['labels'] != -100])
    print('Supervised answer:', text[:500])
    assert objects_to_answer(rows[0]['objects']) in text
    batch = move_batch(collate([sample], processor.tokenizer.pad_token_id))
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
        loss = model(**batch).loss
    assert torch.isfinite(loss), loss
    print('Preflight forward loss:', float(loss), flush=True)
    del batch


def illustrations(name):
    from PIL import Image, ImageDraw
    from scripts.labcommon import read_jsonl
    from scripts.score import parse_output
    pred = {r['image_id']: r['raw'] for r in read_jsonl(SAVE/'predictions'/f'{name.lower()}_test.jsonl')}
    folder = Path('report/images')
    folder.mkdir(parents=True, exist_ok=True)
    rows = read_rows('test')
    for idx in range(6):
        row = rows[idx * max(1, len(rows)//6)]
        with Image.open(Path('data')/row['image_path']) as source:
            im = source.convert('RGB')
        draw = ImageDraw.Draw(im)
        for o in row['objects']:
            box = yolo_to_box(o['label'], o['yolo'])['bbox_2d']
            xy = [box[0]*im.width/1000,box[1]*im.height/1000,box[2]*im.width/1000,box[3]*im.height/1000]
            draw.rectangle(xy, outline='lime', width=2)
            draw.text((xy[0],max(0,xy[1]-12)), 'GT '+o['label'], fill='lime')
        for o in parse_output(pred[row['image_id']]) or []:
            box = o['bbox']
            xy = [box[0]*im.width/1000,box[1]*im.height/1000,box[2]*im.width/1000,box[3]*im.height/1000]
            draw.rectangle(xy, outline='red', width=2)
            draw.text((xy[0],xy[1]), 'Pred '+o['label'], fill='red')
        im.save(folder/f'{name.lower()}_{idx+1}.jpg')
    dump(folder/'captions.json', {'setup':name,'green':'Ground truth','red':'Prediction',
        'image_ids':[rows[i*max(1,len(rows)//6)]['image_id'] for i in range(6)]})


def report():
    from scripts.labcommon import read_jsonl
    from scripts.score import evaluate, parse_output
    ev = {p.stem: json.loads(p.read_text()) for p in sorted(Path('report/evidence').glob('*.json'))}
    meta = json.loads(Path('data/splits/meta.json').read_text())
    arch = json.loads(Path('work/architecture.json').read_text()) if Path('work/architecture.json').exists() else {}
    lines = ['# Báo cáo Lab Day 10', '', f'Họ tên: **{STUDENT_NAME}**',
        f'Mã học viên: **{STUDENT_ID}**', f'`dataset_version`: `{meta["dataset_version"]}`', '',
        '## Bảng tổng hợp', '',
        '| Mã | Tham số train | VRAM GB | Phút train | F1 | CI95 | FP/ảnh | pothole | crack | manhole |',
        '|---|---:|---:|---:|---:|---|---:|---:|---:|---:|']
    for key, d in ev.items():
        m=d.get('meta',{}); c=d['f1_ci95']
        cells = [key, str(m.get('trainable_params',0)),f'{m.get("peak_vram_gb",0):.3f}',
            f'{m.get("train_minutes",0):.2f}',f'{d["micro"]["f1"]:.4f}',f'[{c[0]:.4f}, {c[1]:.4f}]',
            f'{d["fp_per_image"]:.4f}'] + [f'{d["per_class"][x]["f1"]:.4f}' for x in meta['classes']]
        lines.append('| '+' | '.join(cells)+' |')
    lines += ['', 'E0 không huấn luyện; thời gian train bằng 0. Các số chỉ xuất hiện khi có evidence thật.',
        'Batch train 1, gradient accumulation 8, effective batch 8; NF4 trên LLM, visual và lm_head không lượng tử hóa.',
        'Không dùng tập test để chọn setup: chọn bằng 40 ảnh val cố định; tập con nhỏ giới hạn độ tin cậy.', '',
        '## 10 câu hỏi', '', '### Câu 1 — Kiến trúc và khối được cập nhật',
        f'Số tham số dense đo từ model thật (khôi phục kích thước Params4bit): `{arch}`. Visual bao gồm projector; không cộng projector lần thứ hai.',
        'E1: LoRA attention/MLP của LLM. E2: E1 và toàn bộ merger/DeepStack projector. E3: E2 và LoRA attention/MLP của ViT.', '']
    if 'e0' in ev:
        raw=read_jsonl(SAVE/'predictions/e0_test.jsonl')
        invalid=[r for r in raw if parse_output(r['raw']) is None]
        lines += ['### Câu 2 — E0', f'parse_valid_rate = {ev["e0"]["parse_valid_rate"]:.4f}; JSON hỏng: {len(invalid)}/{len(raw)} ảnh.',
            'Ví dụ output nguyên bản:', '```json', (invalid or raw)[0]['raw'], '```',
            'FP/ảnh, recall và ảnh minh họa cho biết mức bịa/thừa box và bỏ sót. Không khẳng định sai thang tọa độ chỉ từ F1 thấp.', '']
        wrong=evaluate(read_rows('test'),{r['image_id']:r['raw'] for r in raw},coord='pixel')
        dump('work/e0_wrong_scale_diagnostic.json',wrong)
        lines += ['### Câu 3 — Thang tọa độ',
            f'Cùng raw E0, chấm norm1000: F1={ev["e0"]["micro"]["f1"]:.4f}; cố ý chấm pixel: F1={wrong["micro"]["f1"]:.4f}. Đây là diagnostic, không phải evidence setup mới.',
            'Thang 0–1000 biểu diễn vị trí tương đối; với ảnh 640×360, pixel x = x_norm×640/1000 và pixel y = y_norm×360/1000. Dùng sai thang làm lệch box và IoU.', '']
    else:
        lines += ['### Câu 2 — E0','[Cần bổ sung: E0 chưa hoàn tất]','','### Câu 3 — Thang tọa độ','[Cần bổ sung: chưa có raw để kiểm chứng]','']
    lines += ['### Câu 4 — Mask loss',
        'Mask prompt, token ảnh và padding bằng -100. Chỉ tối ưu JSON đáp án và token kết thúc. Kiểm tra prompt là tiền tố của full chat, 220 image tokens và giải mã phần supervised. Nếu học prompt/ảnh, mục tiêu bị pha loãng bởi token không phải đầu ra cần dự đoán.', '',
        '### Câu 5 — Projector và lượng tử hóa',
        'Projector train đầy đủ phải có tham số float32 cho optimizer/gradient ổn định trong AMP fp16. Forward chạy autocast fp16. Đây là thiết lập của lab, không phải mọi bài toán đều bắt buộc float32. NF4 Params4bit không hỗ trợ cập nhật dense như tham số float; bỏ visual và lm_head khỏi lượng tử hóa. Projector được lưu/nạp riêng ngoài PEFT adapter.', '',
        '### Câu 6 — Split theo phiên quay',
        'Frame liền kề có nội dung gần trùng. Chia ngẫu nhiên có thể đưa cùng phiên vào train/test, làm đánh giá lạc quan. Kiểm tra giao tập session bằng rỗng; không đổi split của đề.',
        '```json',json.dumps(meta['counts'],ensure_ascii=False,indent=2),'```','',
        '### Câu 7 — So sánh E1/E2/E3',
        'Đối chiếu bảng trên theo F1, CI95, VRAM, thời gian; chỉ kết luận từ setup đã chạy đủ. Khoảng tin cậy từng setup không phải kiểm định paired difference.', '',
        '### Câu 8 — Test nhỏ và khoảng tin cậy',
        'Scorer bootstrap theo ảnh, 1.000 lần. Độ rộng CI được thể hiện trong bảng. Frame còn tương quan trong phiên nên CI theo ảnh có thể lạc quan; chưa có bootstrap theo session. Chênh lệch nhỏ không đủ chứng minh setup tốt hơn.', '',
        '### Câu 9 — Phân tích lỗi theo lớp',
        'F1 từng lớp trong bảng. Box nhỏ nhạy với sai tọa độ; số lượng lớp khác nhau giữa train/val/test. Nguyên nhân này là giả thuyết, cần đối chiếu ảnh thay vì coi là kết luận đã chứng minh.', '',
        '### Câu 10 — E4',
        'Chọn data_size: 10% và 100% trên cùng setup tốt nhất theo val. Giữ seed=42, rank=16, optimizer, prompt, độ phân giải, max_steps=300, effective batch=8. 100% dùng lại baseline đúng cấu hình; 10% lấy prefix sau shuffle seed=42. Với bước tối ưu cố định, tập nhỏ được lặp nhiều lần hơn; đây là một giới hạn diễn giải.', '',
        '## Ảnh minh họa', '', 'Box xanh: nhãn thật. Box đỏ: dự đoán. Ảnh được lấy theo chỉ số cố định, không chỉ chọn ảnh đẹp.']
    for p in sorted(Path('report/images').glob('*.jpg')):
        lines += ['',f'![{p.stem}]({p.as_posix()})']
    missing=[k for k in ('e0','e1','e2','e3') if k not in ev]
    lines += ['', '## Nhìn lại', f'Thiếu evidence: {missing or "không"}. E5 GRPO không thực hiện (tùy chọn).',
        'Test/inference/training thật mới chứng minh kết quả; các self-check không thay thế thí nghiệm.']
    Path('REPORT.md').write_text('\n'.join(lines)+'\n',encoding='utf-8')
    backup()


def run_all(existing=None):
    import torch
    from scripts.score import evaluate
    torch.manual_seed(42)
    torch.cuda.manual_seed_all(42)
    meta=json.loads(Path('data/splits/meta.json').read_text())
    splits={s:read_rows(s) for s in ('train','val','test')}
    groups={s:{r['session'] for r in rows} for s,rows in splits.items()}
    assert not groups['train'] & groups['val'] and not groups['train'] & groups['test'] and not groups['val'] & groups['test']
    SAVE.mkdir(parents=True,exist_ok=True)
    dump(SAVE/'provenance.json', dict(student=STUDENT_NAME,student_id=STUDENT_ID,
        dataset_version=meta['dataset_version'], prompt=PROMPT,
        source_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),
        git_commit=subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()))
    backup()
    if not Path('report/evidence/e0.json').exists():
        model, processor = existing if existing else load_base()
        del existing
        preflight(model, processor)
        torch.cuda.reset_peak_memory_stats()
        path,_=infer(model,processor,'E0')
        score(path,'E0',dict(trainable_params=0,peak_vram_gb=torch.cuda.max_memory_allocated()/2**30,train_minutes=0))
        del model,processor
        gc.collect(); torch.cuda.empty_cache()
    val_scores={}
    for setup in ('E1','E2','E3'):
        name=setup
        evidence=Path(f'report/evidence/{name.lower()}.json')
        if not evidence.exists():
            model,processor=load_base()
            model,run=train(model,processor,setup,name)
            del model,processor
            gc.collect(); torch.cuda.empty_cache()
            # Fresh base + saved adapter/projector tests the real checkpoint contract.
            model,processor=load_trained(setup,name)
            path,_=infer(model,processor,name)
            score(path,name,run)
            _,val_pred=infer(model,processor,name,'val',limit=40)
            val=evaluate(splits['val'][:40],val_pred)
            dump(SAVE/'runs'/name.lower()/'val40.json',val)
            del model,processor
            gc.collect(); torch.cuda.empty_cache()
        if not (SAVE/'runs'/name.lower()/'val40.json').exists():
            model,processor=load_trained(setup,name)
            _,val_pred=infer(model,processor,name,'val',limit=40)
            dump(SAVE/'runs'/name.lower()/'val40.json',evaluate(splits['val'][:40],val_pred))
            del model,processor
            gc.collect(); torch.cuda.empty_cache()
        val=json.loads((SAVE/'runs'/name.lower()/'val40.json').read_text())
        val_scores[name]=val['micro']['f1']
        illustrations(name)
        report()
    best=max(val_scores,key=val_scores.get)
    dump('work/selection.json',dict(val40_scores=val_scores,best=best))
    best_evidence=json.loads(Path(f'report/evidence/{best.lower()}.json').read_text())
    best_evidence['name']=f'E4_data_size_1.0 ({best})'
    dump('report/evidence/e4_data_size_1.0.json',best_evidence)
    name='E4_data_size_0.1'
    if not Path(f'report/evidence/{name.lower()}.json').exists():
        model,processor=load_base()
        model,run=train(model,processor,best,name,fraction=0.1)
        del model,processor
        gc.collect(); torch.cuda.empty_cache()
        model,processor=load_trained(best,name)
        path,_=infer(model,processor,name)
        score(path,name,run)
        del model,processor
        gc.collect(); torch.cuda.empty_cache()
    report()
    result=subprocess.run(['python','scripts/check_submission.py'],capture_output=True,text=True)
    print(result.stdout,flush=True)
    dump(SAVE/'submission_check.json',dict(returncode=result.returncode,output=result.stdout))
    print('GPU EXPERIMENTS FINISHED',flush=True)


if __name__ == '__main__':
    run_all()


## 3. E0: zero-shot

In [ ]:
import sys, importlib
sys.path.insert(0,str(ROOT)) if str(ROOT) not in sys.path else None
import student_lab as lab
importlib.reload(lab)
model,processor=lab.load_base()
lab.preflight(model,processor)
torch.cuda.reset_peak_memory_stats()
path,predictions=lab.infer(model,processor,'E0')
lab.score(path,'E0',{'trainable_params':0,'peak_vram_gb':torch.cuda.max_memory_allocated()/2**30,'train_minutes':0})
del model,processor
import gc
gc.collect(); torch.cuda.empty_cache()


In [ ]:
print(Path('report/evidence/e0.json').read_text())


## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
# 4. Encoding chat + loss mask + collate: mã thực hiện đầy đủ.
import torch
from transformers import AutoProcessor
from student_lab import MODEL_ID, PROMPT, user_message, read_rows, objects_to_answer
def encode_example(processor, row):
    import torch
    user = user_message(row)
    prompt = processor.apply_chat_template([user], tokenize=True,
        add_generation_prompt=True, return_dict=True, return_tensors='pt')
    full = processor.apply_chat_template([user,
        {'role': 'assistant', 'content': [{'type': 'text', 'text': objects_to_answer(row['objects'])}]}],
        tokenize=True, add_generation_prompt=False, return_dict=True, return_tensors='pt')
    n = prompt['input_ids'].shape[1]
    assert torch.equal(full['input_ids'][:, :n], prompt['input_ids']), 'Chat prefix mismatch'
    full['labels'] = full['input_ids'].clone()
    full['labels'][:, :n] = -100
    assert (full['labels'] != -100).any(), 'No supervised tokens'
    return dict(full)

def collate(batch, pad_token_id):
    import torch
    import torch.nn.functional as F
    n = max(x['input_ids'].shape[1] for x in batch)
    out = {}
    for key in batch[0]:
        if key in ('pixel_values', 'image_grid_thw'):
            out[key] = torch.cat([x[key] for x in batch], dim=0)
        elif key in ('input_ids', 'attention_mask', 'labels', 'mm_token_type_ids', 'token_type_ids'):
            pad = -100 if key == 'labels' else pad_token_id if key == 'input_ids' else 0
            out[key] = torch.cat([F.pad(x[key], (0, n-x[key].shape[1]), value=pad) for x in batch])
        else:
            raise ValueError(f'Unsupported processor field: {key}')
    return out

processor = AutoProcessor.from_pretrained(MODEL_ID,min_pixels=1024,max_pixels=245760)
processor.tokenizer.padding_side='left'
sample=encode_example(processor,read_rows('train')[0])
image_token=processor.tokenizer.convert_tokens_to_ids('<|image_pad|>')
assert int((sample['input_ids']==image_token).sum())==220
supervised=processor.tokenizer.decode(sample['labels'][sample['labels']!=-100])
assert objects_to_answer(read_rows('train')[0]['objects']) in supervised
batch_check=collate([sample,sample],processor.tokenizer.pad_token_id)
assert batch_check['input_ids'].shape[0]==2
print('220 image tokens; chat prefix/mask và collate PASS')
print('Chỉ tính loss trên:',supervised)
del batch_check,sample


In [ ]:
%%writefile finalize_lab.py
"""Queue after E0-E4: validate real artifacts, finish report, export review ZIP."""
import hashlib
import json
import math
import re
import shutil
import subprocess
import zipfile
from pathlib import Path

EXPECTED = ('e0', 'e1', 'e2', 'e3', 'e4_data_size_0.1', 'e4_data_size_1.0')


def check_evidence(evidence, version, n_images):
    errors = []
    for name in EXPECTED:
        d = evidence.get(name)
        if d is None:
            errors.append(f'{name}: missing evidence')
            continue
        for key, expected in (('dataset_version', version), ('split', 'test'),
                              ('n_images', n_images), ('missing_predictions', 0), ('score_version', 1)):
            if d.get(key) != expected:
                errors.append(f'{name}: {key} must equal {expected}')
        if name != 'e0':
            for key in ('trainable_params', 'peak_vram_gb', 'train_minutes'):
                v = d.get('meta', {}).get(key)
                if not isinstance(v, (int, float)) or isinstance(v, bool) or not math.isfinite(v) or v <= 0:
                    errors.append(f'{name}: invalid {key}')
        f1 = d.get('micro', {}).get('f1')
        if not isinstance(f1, (int, float)) or isinstance(f1, bool) or not 0 <= f1 <= 1:
            errors.append(f'{name}: invalid F1')
        ci = d.get('f1_ci95')
        if not isinstance(ci, list) or len(ci) != 2 or not all(isinstance(x, (int, float)) for x in ci) or not 0 <= ci[0] <= ci[1] <= 1:
            errors.append(f'{name}: invalid CI95')
    return errors


def comparison(a, b, label):
    delta = b['micro']['f1'] - a['micro']['f1']
    overlap = max(a['f1_ci95'][0], b['f1_ci95'][0]) <= min(a['f1_ci95'][1], b['f1_ci95'][1])
    return (f'{label}: ΔF1={delta:+.4f}; ΔVRAM={b["meta"]["peak_vram_gb"]-a["meta"]["peak_vram_gb"]:+.3f} GB; '
            f'Δthời gian train={b["meta"]["train_minutes"]-a["meta"]["train_minutes"]:+.2f} phút. '
            f'CI95 {"chồng lấn" if overlap else "không chồng lấn"}; đây không phải kiểm định paired difference.')


def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def replace_answer(text, question, extra):
    pattern = rf'(### Câu {question}[^\n]*\n)(.*?)(?=\n### Câu |\n## |\Z)'
    return re.sub(pattern, lambda m: m.group(1) + m.group(2).rstrip() + '\n\n' + extra + '\n', text, count=1, flags=re.S)


def examples(lab, name):
    from PIL import Image, ImageDraw
    from scripts.labcommon import read_jsonl
    from scripts.score import parse_output, _gt_boxes, _match
    predictions = {r['image_id']: r['raw'] for r in read_jsonl(lab.SAVE/'predictions'/f'{name}_test.jsonl')}
    measured = []
    for row in lab.read_rows('test'):
        parsed = parse_output(predictions[row['image_id']])
        per, ious = _match(_gt_boxes(row), parsed or [])
        tp, fp, fn = (sum(v[k] for v in per.values()) for k in range(3))
        f1 = 2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0
        measured.append((row, parsed or [], dict(image_id=row['image_id'], tp=tp, fp=fp, fn=fn,
                        f1=f1, valid_json=parsed is not None, mean_iou=sum(ious)/len(ious) if ious else 0)))
    good = sorted((x for x in measured if x[2]['tp'] > 0), key=lambda x: x[2]['f1'], reverse=True)[:3]
    used = {x[0]['image_id'] for x in good}
    bad = sorted((x for x in measured if x[0]['image_id'] not in used),
                 key=lambda x: x[2]['fp']+x[2]['fn'], reverse=True)[:6-len(good)]
    folder = Path('report/examples'); folder.mkdir(parents=True, exist_ok=True)
    captions = []
    for i, (row, preds, counts) in enumerate(good+bad, 1):
        with Image.open(Path('data')/row['image_path']) as src:
            im = src.convert('RGB')
        draw = ImageDraw.Draw(im)
        for lab_name, box, color in [(x['label'], x['bbox'], 'lime') for x in _gt_boxes(row)] + [(x['label'], x['bbox'], 'red') for x in preds]:
            xy = [box[0]*im.width/1000, box[1]*im.height/1000, box[2]*im.width/1000, box[3]*im.height/1000]
            draw.rectangle(xy, outline=color, width=2)
            draw.text((xy[0], xy[1]), lab_name, fill=color)
        banner = Image.new('RGB', (im.width, im.height+24), 'black'); banner.paste(im, (0, 0))
        ImageDraw.Draw(banner).text((4, im.height+4), f'{name} | green=GT red=Pred | TP={counts["tp"]} FP={counts["fp"]} FN={counts["fn"]}', fill='white')
        path = folder/f'{name}_{i}.jpg'; banner.save(path)
        captions.append(dict(counts, path=path.as_posix(), selection='highest F1 with TP' if i <= len(good) else 'most FP+FN'))
    lab.dump(folder/'captions.json', captions)
    return captions


def finalize():
    import student_lab as lab
    from scripts.labcommon import read_jsonl
    meta = json.loads(Path('data/splits/meta.json').read_text())
    evidence = {p.stem: json.loads(p.read_text()) for p in Path('report/evidence').glob('*.json')}
    errors = check_evidence(evidence, meta['dataset_version'], meta['counts']['test']['images'])
    lab.dump(lab.SAVE/'finalization_status.json', {'experiments_complete': not errors, 'errors': errors})
    if errors:
        raise RuntimeError('Không tạo báo cáo hoàn tất từ evidence thiếu/sai:\n'+'\n'.join(errors))
    runs = {}
    for name in ('e1', 'e2', 'e3', 'e4_data_size_0.1'):
        folder = lab.SAVE/'runs'/name
        run = json.loads((folder/'run.json').read_text())
        assert run['completed_steps'] == run['max_steps'] == 300, f'{name}: training incomplete'
        for key in ('trainable_params', 'peak_vram_gb', 'train_minutes'):
            assert math.isclose(run[key], evidence[name]['meta'][key]), f'{name}: metadata mismatch {key}'
        if run['setup'] in ('E2', 'E3'):
            assert (folder/'projector.pt').is_file(), f'{name}: projector checkpoint missing'
        runs[name] = run
    assert runs['e2']['trainable_params']-runs['e1']['trainable_params'] == 109105152
    assert runs['e3']['trainable_params'] > runs['e2']['trainable_params']
    selection = json.loads(Path('work/selection.json').read_text())
    lab.dump('report/selection.json', selection)
    if (lab.SAVE/'provenance.json').exists():
        shutil.copy2(lab.SAVE/'provenance.json', 'report/provenance.json')
    best = selection['best'].lower()
    assert runs['e4_data_size_0.1']['setup'].lower() == best
    for key in ('micro', 'per_class', 'f1_ci95'):
        assert evidence['e4_data_size_1.0'][key] == evidence[best][key], f'E4 baseline mismatch {key}'
    raw_dir = Path('report/raw_predictions'); raw_dir.mkdir(parents=True, exist_ok=True)
    for name in ('e0', 'e1', 'e2', 'e3', 'e4_data_size_0.1'):
        src = lab.SAVE/'predictions'/f'{name}_test.jsonl'
        records = read_jsonl(src)
        assert len(records) == len({r['image_id'] for r in records}) == meta['counts']['test']['images']
        shutil.copy2(src, raw_dir/src.name)
    captions = examples(lab, best)
    lab.report()
    text = Path('REPORT.md').read_text(encoding='utf-8')
    e0 = evidence['e0']['micro']
    text = replace_answer(text, 2, f'Hai nhóm lỗi đo bằng scorer: bỏ sót FN={e0["fn"]}, dự đoán không khớp FP={e0["fp"]}; TP={e0["tp"]}. FP có thể do sai vị trí hoặc sai lớp, không đồng nhất mọi FP với hallucination.')
    text = replace_answer(text, 5, "Đã kiểm chứng Transformers 5.19.0 khớp prefix đầy đủ: dùng llm_int8_skip_modules=['model.visual','lm_head']; tên ngắn 'visual' không giữ được ViT/projector dạng float.")
    text = replace_answer(text, 7, '\n\n'.join([comparison(evidence['e1'], evidence['e2'], 'E2 so với E1'), comparison(evidence['e2'], evidence['e3'], 'E3 so với E2'), f'Setup chọn trên 40 ảnh val cố định: {selection["best"]}; F1 val: {selection["val40_scores"]}. Lựa chọn này không sử dụng test.']))
    text = replace_answer(text, 8, 'Độ rộng CI95 thật: '+', '.join(f'{name}={evidence[name]["f1_ci95"][1]-evidence[name]["f1_ci95"][0]:.4f}' for name in ('e0','e1','e2','e3'))+'.')
    worst = min(meta['classes'], key=lambda c: evidence[best]['per_class'][c]['f1'])
    cls = evidence[best]['per_class'][worst]
    text = replace_answer(text, 9, f'Ở {best}, lớp F1 thấp nhất là {worst}: F1={cls["f1"]:.4f}, TP={cls["tp"]}, FP={cls["fp"]}, FN={cls["fn"]}. Xem sáu ảnh có TP và/hoặc nhiều lỗi dưới đây; nguyên nhân cần học viên đối chiếu ảnh trước khi kết luận.')
    text = replace_answer(text, 10, comparison(evidence['e4_data_size_1.0'], evidence['e4_data_size_0.1'], '10% so với 100%')+f' Số ảnh train: {runs["e4_data_size_0.1"]["train_images"]} và {runs[best]["train_images"]}.')
    text += '\n## Sáu ảnh đối chiếu có số TP/FP/FN\n\nChọn tối đa ba ảnh có TP với F1 cao, các ảnh còn lại có nhiều FP+FN; đây là minh họa có chủ đích, không dùng để ước lượng chất lượng tổng thể.\n'
    for c in captions:
        text += f'\n![{c["image_id"]}]({c["path"]})\n\nTP={c["tp"]}, FP={c["fp"]}, FN={c["fn"]}, F1={c["f1"]:.4f}, JSON hợp lệ={c["valid_json"]}.\n'
    Path('REPORT.md').write_text(text, encoding='utf-8')
    train_dir = Path('report/training'); train_dir.mkdir(parents=True, exist_ok=True)
    for name in runs:
        for filename in ('run.json', 'loss.jsonl', 'val40.json', 'amp_events.jsonl'):
            src = lab.SAVE/'runs'/name/filename
            if src.exists():
                shutil.copy2(src, train_dir/f'{name}_{filename}')
    result = subprocess.run(['python','scripts/check_submission.py'], capture_output=True, text=True)
    rubric = {'e1_parse_valid_rate_at_least_0.95': evidence['e1']['parse_valid_rate'] >= 0.95,
              'e1_f1_improvement_at_least_0.10': evidence['e1']['micro']['f1']-evidence['e0']['micro']['f1'] >= 0.10}
    status = {'experiments_complete': True, 'submission_ready': result.returncode == 0 and all(rubric.values()),
              'rubric': rubric, 'checker_returncode': result.returncode, 'checker_output': result.stdout,
              'student_name': lab.STUDENT_NAME, 'student_id': lab.STUDENT_ID}
    lab.dump('report/finalization_status.json', status)
    print(result.stdout); print(json.dumps(status, ensure_ascii=False, indent=2))
    lab.backup()
    shutil.copy2('finalize_lab.py', lab.SAVE/'finalize_lab.py')
    files = [p for folder in ('configs','scripts','tests','report') for p in Path(folder).rglob('*')
             if p.is_file() and '__pycache__' not in p.parts and p.suffix != '.pyc' and p.name != 'file_hashes.json']
    files += [Path(p) for p in ('README.md','GUIDE.md','RUBRIC.md','REPORT.md','requirements-colab.txt',
              'pytest.ini','lab_day10.ipynb','student_lab.py','student_checks.py','finalize_lab.py') if Path(p).exists()]
    lab.dump('report/file_hashes.json', {p.as_posix(): digest(p) for p in files})
    files.append(Path('report/file_hashes.json'))
    archive = lab.SAVE/f'Day10_{lab.STUDENT_ID}_review.zip'
    with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as z:
        for p in files:
            z.write(p, p.as_posix())
    lab.dump(lab.SAVE/'finalization_status.json', status)
    shutil.copy2('report/file_hashes.json', lab.SAVE/'report/file_hashes.json')
    print('REVIEW ZIP:', archive, '— kiểm tra submission_ready trước khi nộp.')
    return status


if __name__ == '__main__':
    finalize()


In [ ]:
%%writefile student_checks.py
"""Checks owned by the student; instructor tests stay unchanged."""
import json
import unittest


class Contracts(unittest.TestCase):
    def test_amp_overflow_keeps_weights_and_reduces_scale(self):
        import torch
        from student_lab import amp_optimizer_step
        p = torch.nn.Parameter(torch.tensor([1.0]))
        opt = torch.optim.AdamW([p], lr=0.1, weight_decay=0)
        scaler = torch.amp.GradScaler('cpu', init_scale=8)
        scaler.scale(p.sum()).backward()
        p.grad.fill_(float('inf'))
        updated, norm = amp_optimizer_step(opt, [p], scaler)
        self.assertFalse(updated)
        self.assertEqual(p.item(), 1.0)
        self.assertEqual(len(opt.state), 0)
        self.assertEqual(scaler.get_scale(), 4.0)
        self.assertIsNone(p.grad)
        scaler.scale(p.sum()).backward()
        updated, norm = amp_optimizer_step(opt, [p], scaler)
        self.assertTrue(updated)
        self.assertTrue(torch.isfinite(p).all())
        self.assertLess(p.item(), 1.0)

    def test_finalization_refuses_incomplete_evidence(self):
        from finalize_lab import check_evidence, EXPECTED
        errors = check_evidence({}, 'test-version', 311)
        self.assertEqual(len(errors), len(EXPECTED))
        self.assertTrue(all('missing evidence' in x for x in errors))

    def test_grounding_coordinates_and_empty_answer(self):
        import student_lab as lab
        self.assertEqual(lab.yolo_to_box('crack', [0.5, 0.5, 0.2, 0.4]),
                         {'label': 'crack', 'bbox_2d': [400, 300, 600, 700]})
        self.assertEqual(lab.objects_to_answer([]), '[]')
        boxes = json.loads(lab.objects_to_answer([
            {'label': 'pothole', 'yolo': [0.5, 0.8, 0.2, 0.1]},
            {'label': 'crack', 'yolo': [0.5, 0.2, 0.2, 0.1]}]))
        self.assertEqual(boxes[0]['label'], 'crack')

    def test_collate_preserves_image_tensors_and_masks_padding(self):
        import torch
        import student_lab as lab
        a = {'input_ids': torch.tensor([[1, 2]]), 'attention_mask': torch.ones(1, 2, dtype=torch.long),
             'labels': torch.tensor([[-100, 2]]), 'pixel_values': torch.ones(4, 3),
             'image_grid_thw': torch.tensor([[1, 2, 2]])}
        b = {'input_ids': torch.tensor([[1, 2, 3]]), 'attention_mask': torch.ones(1, 3, dtype=torch.long),
             'labels': torch.tensor([[-100, 2, 3]]), 'pixel_values': torch.ones(8, 3),
             'image_grid_thw': torch.tensor([[1, 2, 4]])}
        out = lab.collate([a, b], 0)
        self.assertEqual(out['labels'].tolist(), [[-100, 2, -100], [-100, 2, 3]])
        self.assertEqual(out['pixel_values'].shape, (12, 3))
        self.assertEqual(out['image_grid_thw'].shape, (2, 3))
        self.assertEqual(out['attention_mask'].tolist(), [[1, 1, 0], [1, 1, 1]])


if __name__ == '__main__':
    unittest.main()


## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
# 5. LoRA LLM / projector / ViT: mã thực hiện đầy đủ.
import re,json
from pathlib import Path
def is_projector(name):
    return 'visual.merger' in name or 'visual.deepstack_merger_list' in name

def apply_setup(model, setup, rank=16):
    import torch
    from peft import LoraConfig, get_peft_model
    llm = r'.*language_model\.layers\.\d+\.(?:self_attn\.(?:q|k|v|o)_proj|mlp\.(?:gate|up|down)_proj)'
    vit = r'.*visual\.blocks\.\d+\.(?:attn\.(?:qkv|proj)|mlp\.(?:linear_fc1|linear_fc2))'
    pattern = f'(?:{llm}|{vit})' if setup == 'E3' else llm
    matched = [n for n, _ in model.named_modules() if re.fullmatch(pattern, n)]
    expected = 348 if setup == 'E3' else 252
    assert len(matched) == expected, (len(matched), expected, matched[:5])
    for p in model.parameters():
        p.requires_grad = False
    model.enable_input_require_grads()
    model = get_peft_model(model, LoraConfig(r=rank, lora_alpha=2*rank,
        lora_dropout=0.05, target_modules=pattern, task_type='CAUSAL_LM'))
    if setup in ('E2', 'E3'):
        for name, p in model.named_parameters():
            if is_projector(name):
                assert p.__class__.__name__ != 'Params4bit', 'Projector is quantized'
                p.data = p.data.to(torch.float32)
                p.requires_grad = True
    # PEFT adapters must be float32 for fp16 GradScaler unscale.
    for name, p in model.named_parameters():
        if p.requires_grad:
            p.data = p.data.float()
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
    model.config.use_cache = False
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(setup, 'LoRA modules:', len(matched), 'trainable_params:', n, flush=True)
    return model, n

# Hàm này được train() gọi; không nạp thêm model trong lúc pipeline đang chạy.
for setup in ('e1','e2','e3'):
    path=lab.SAVE/'runs'/setup/'run.json'
    if path.exists():
        run=json.loads(path.read_text())
        print(setup,run['setup'],'trainable_params=',run['trainable_params'])
    else:
        print(setup,'chưa có checkpoint hoàn tất; không thay bằng số giả')


## 6. Vòng huấn luyện

In [ ]:
# 6. Vòng train thật: AdamW, AMP, accumulation, resume và checkpoint Drive.
import gc,random,time,math
from student_lab import SAVE,dump,read_rows
def move_batch(batch):
    import torch
    return {k: v.to('cuda', dtype=torch.float16 if k == 'pixel_values' else v.dtype)
            for k, v in batch.items()}

def checkpoint(model, folder, state):
    import torch
    folder.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(folder / 'adapter')
    extra = {n: p.detach().cpu() for n, p in model.named_parameters()
             if p.requires_grad and is_projector(n)}
    if extra:
        torch.save(extra, folder / 'projector.pt')
    tmp = folder / 'trainer.tmp.pt'
    torch.save(state, tmp)
    tmp.replace(folder / 'trainer.pt')
    dump(folder / 'run.json', state['run'])

def restore_projector(model, folder):
    import torch
    path = Path(folder) / 'projector.pt'
    if path.exists():
        values = torch.load(path, map_location='cpu', weights_only=True)
        named = dict(model.named_parameters())
        for name, value in values.items():
            assert name in named, f'Missing projector key {name}'
            named[name].data = value.to(device=named[name].device, dtype=torch.float32)
        print('Restored projector tensors:', len(values), flush=True)

def amp_optimizer_step(opt, params, scaler):
    """Never update weights with non-finite gradients; lower scale and retry."""
    import torch
    scaler.unscale_(opt)
    norm = torch.nn.utils.clip_grad_norm_(params, 1.0, error_if_nonfinite=False)
    if not torch.isfinite(norm):
        scaler.update(new_scale=scaler.get_scale() * scaler.get_backoff_factor())
        opt.zero_grad(set_to_none=True)
        return False, float(norm)
    scaler.step(opt)
    scaler.update()
    opt.zero_grad(set_to_none=True)
    return True, float(norm)

def train(model, processor, setup, name, rank=16, fraction=1.0, steps=300):
    import torch
    from transformers import get_cosine_schedule_with_warmup
    folder = SAVE / 'runs' / name.lower()
    folder.mkdir(parents=True, exist_ok=True)
    model, nparams = apply_setup(model, setup, rank)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=2e-4, weight_decay=0)
    sched = get_cosine_schedule_with_warmup(opt, math.ceil(steps*0.05), steps)
    scaler = torch.amp.GradScaler('cuda')
    rows = read_rows('train')
    random.Random(42).shuffle(rows)
    rows = rows[:max(1, int(len(rows)*fraction))]
    run = dict(name=name, setup=setup, rank=rank, train_fraction=fraction, seed=42,
               trainable_params=nparams, batch_size=1, grad_accum=8, max_steps=steps,
               optimizer='torch.optim.AdamW', lr=2e-4, completed_steps=0,
               train_minutes=0.0, peak_vram_gb=0.0, train_images=len(rows))
    start_step = 0
    saved = folder / 'trainer.pt'
    if saved.exists():
        state = torch.load(saved, map_location='cpu', weights_only=False)
        old = state['run']
        for key in ('name', 'setup', 'rank', 'train_fraction', 'max_steps'):
            assert old[key] == run[key], f'Resume mismatch: {key}'
        from peft import load_peft_weights, set_peft_model_state_dict
        set_peft_model_state_dict(model, load_peft_weights(str(folder/'adapter'), device='cpu'))
        restore_projector(model, folder)
        opt.load_state_dict(state['optimizer'])
        sched.load_state_dict(state['scheduler'])
        scaler.load_state_dict(state['scaler'])
        run = old
        start_step = old['completed_steps']
        torch.set_rng_state(state['rng'])
        torch.cuda.set_rng_state_all(state['cuda_rng'])
        print('Resume', name, start_step, flush=True)
        del state
    torch.cuda.reset_peak_memory_stats()
    begin = time.perf_counter()
    previous_minutes = run['train_minutes']
    cache = {}
    model.train()
    opt.zero_grad(set_to_none=True)
    log = folder / 'loss.jsonl'
    step, overflow_retries = start_step, 0
    while step < steps:
        loss_sum = 0.0
        for micro in range(8):
            idx = (step*8 + micro) % len(rows)
            if idx not in cache:
                cache[idx] = encode_example(processor, rows[idx])
            batch = move_batch(collate([cache[idx]], processor.tokenizer.pad_token_id))
            with torch.autocast('cuda', dtype=torch.float16):
                loss = model(**batch).loss
            if not torch.isfinite(loss):
                raise FloatingPointError(f'{name} non-finite loss at {step+1}')
            loss_sum += float(loss.detach()) / 8
            scaler.scale(loss / 8).backward()
            del batch, loss
        scale_before = scaler.get_scale()
        updated, norm = amp_optimizer_step(opt, params, scaler)
        if not updated:
            overflow_retries += 1
            run['amp_overflow_skips'] = run.get('amp_overflow_skips', 0) + 1
            event = dict(event='amp_overflow_retry', step=step+1, retry=overflow_retries,
                         scale_before=scale_before, scale_after=scaler.get_scale(),
                         image_ids=[rows[(step*8+i) % len(rows)]['image_id'] for i in range(8)])
            with (folder / 'amp_events.jsonl').open('a', encoding='utf-8') as f:
                f.write(json.dumps(event)+'\n')
            print(name, json.dumps(event), flush=True)
            if overflow_retries >= 8:
                raise FloatingPointError(f'{name}: gradients remain non-finite after 8 retries; see amp_events.jsonl')
            continue
        overflow_retries = 0
        sched.step()
        run.update(completed_steps=step+1,
            train_minutes=previous_minutes+(time.perf_counter()-begin)/60,
            peak_vram_gb=max(run['peak_vram_gb'], torch.cuda.max_memory_allocated()/2**30))
        record = dict(step=step+1, loss=loss_sum, grad_norm=float(norm),
                      lr=sched.get_last_lr()[0], minutes=run['train_minutes'])
        with log.open('a', encoding='utf-8') as f:
            f.write(json.dumps(record)+'\n')
        if (step+1) % 10 == 0 or step == start_step:
            print(name, json.dumps(record), 'VRAM', round(run['peak_vram_gb'], 2), flush=True)
            dump(folder / 'progress.json', run)
        if (step+1) % 50 == 0 or step+1 == steps:
            checkpoint(model, folder, dict(run=run, optimizer=opt.state_dict(),
                scheduler=sched.state_dict(), scaler=scaler.state_dict(),
                rng=torch.get_rng_state(), cuda_rng=torch.cuda.get_rng_state_all()))
        step += 1
    dump(folder / 'run.json', run)
    del cache, opt, sched, scaler, params
    gc.collect()
    torch.cuda.empty_cache()
    return model, run

# Reload mã mới và giải phóng tham chiếu từ traceback trước khi resume.
import sys,importlib
importlib.reload(lab)
assert hasattr(lab,'amp_optimizer_step'), 'Hãy chạy lại ô writefile student_lab.py bản mới'
for key in ('last_traceback','last_value','last_exc'):
    if hasattr(sys,key):
        setattr(sys,key,None)
globals().pop('model',None)
globals().pop('processor',None)
gc.collect(); torch.cuda.empty_cache()
# Ô pipeline ở trên đã chạy E0–E4. Chỉ resume nếu evidence chưa đủ.
required=['e0','e1','e2','e3','e4_data_size_0.1','e4_data_size_1.0']
missing=[name for name in required if not Path(f'report/evidence/{name}.json').exists()]
if missing:
    print('Resume các phần thiếu:',missing)
    lab.run_all()
else:
    print('Đã có toàn bộ evidence; không train lại.')


## 7. Suy luận và chấm E1–E3

In [ ]:
# 7. Nạp lại adapter/projector, inference và chấm bằng scorer gốc.
import subprocess,json,gc,time
from student_lab import load_base,restore_projector,user_message,read_rows,move_batch,SAVE,backup
def load_trained(setup, name):
    from peft import PeftModel
    model, processor = load_base()
    folder = SAVE / 'runs' / name.lower()
    model = PeftModel.from_pretrained(model, folder / 'adapter')
    restore_projector(model, folder)
    return model, processor

def infer(model, processor, name, split='test', limit=None, batch_size=2):
    import torch
    from scripts.labcommon import read_jsonl
    rows = read_rows(split)
    if limit:
        rows = rows[:limit]
    folder = SAVE / 'predictions'
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / f'{name.lower()}_{split}.jsonl'
    signature = {'dataset_version': json.loads(Path('data/splits/meta.json').read_text())['dataset_version'],
                 'name': name, 'split': split, 'limit': limit, 'prompt': PROMPT,
                 'model_revision': getattr(model.config, '_commit_hash', None)}
    manifest = path.with_suffix('.manifest.json')
    if manifest.exists():
        assert json.loads(manifest.read_text()) == signature, 'Prediction resume provenance mismatch'
    else:
        assert not path.exists(), 'Existing predictions have no provenance manifest'
        dump(manifest, signature)
    done = {}
    if path.exists():
        for r in read_jsonl(path):
            done[r['image_id']] = r['raw']
    remaining = [r for r in rows if r['image_id'] not in done]
    model.eval()
    model.gradient_checkpointing_disable()
    model.config.use_cache = True
    begin = time.perf_counter()
    with path.open('a', encoding='utf-8') as f:
        for start in range(0, len(remaining), batch_size):
            batch = remaining[start:start+batch_size]
            chats = [[user_message(r)] for r in batch]
            inputs = processor.apply_chat_template(chats, tokenize=True, add_generation_prompt=True,
                return_dict=True, return_tensors='pt', padding=True)
            inputs = move_batch(dict(inputs))
            with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
                output = model.generate(**inputs, max_new_tokens=256, do_sample=False)
            answers = processor.batch_decode(output[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)
            for row, raw in zip(batch, answers):
                f.write(json.dumps({'image_id': row['image_id'], 'raw': raw}, ensure_ascii=False)+'\n')
                done[row['image_id']] = raw
            f.flush()
            if start % 20 == 0 or start+batch_size >= len(remaining):
                print(name, split, f'{len(done)}/{len(rows)}', 'minutes', round((time.perf_counter()-begin)/60,2),
                      'sample', answers[0][:160], flush=True)
            del inputs, output
    assert all(r['image_id'] in done for r in rows)
    return path, {r['image_id']: done[r['image_id']] for r in rows}

def score(path, name, run):
    args = ['python', 'scripts/score.py', '--pred', str(path), '--name', name,
            '--out', f'report/evidence/{name.lower()}.json']
    for key in ('trainable_params', 'peak_vram_gb', 'train_minutes'):
        args += ['--meta', f'{key}={run[key]}']
    result = subprocess.run(args, capture_output=True, text=True, check=True)
    print(result.stdout, flush=True)
    backup()

# Raw predictions của các setup đã được pipeline ghi vào Drive.
# Chấm lại từ raw sau khi kiểm tra full test; không sinh dự đoán từ nhãn thật.
for name in ('e0','e1','e2','e3','e4_data_size_0.1'):
    path=SAVE/'predictions'/f'{name}_test.jsonl'
    assert path.exists(),f'Thiếu raw {name}'
    records=[json.loads(x) for x in path.read_text().splitlines() if x.strip()]
    assert len(records)==len({r['image_id'] for r in records})==len(read_rows('test'))
    evidence=json.loads(Path(f'report/evidence/{name}.json').read_text())
    score(path,name.upper(),evidence['meta'])
    print(name,'full test=',len(records),'F1=',evidence['micro']['f1'])


## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
result=subprocess.run(['python','scripts/check_submission.py'],capture_output=True,text=True)
print(result.stdout,result.stderr)
lab.backup()
print('Artifact Drive:',SAVE)
print('E5 GRPO không thực hiện; là phần tùy chọn.')


## 9. Hoàn tất báo cáo, đối chiếu evidence và xuất ZIP
Chạy sau E0–E4. Không tạo số liệu nếu evidence thiếu. Xem submission_ready trước khi nộp.


In [ ]:
import finalize_lab
final_status=finalize_lab.finalize()
print("submission_ready:",final_status["submission_ready"])
